<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")

Total revenue: $8,520.00
Total units: 783


Total revenue is 8,520.00 across 783 units sold, which falls inside the expected 8,000–9,000 range confirming the calculation is correct.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).to_frame()
by_category['pct_of_total'] = (by_category['revenue'] / total_revenue * 100).round(1)
by_category

,revenue,pct_of_total
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


Food dominates the revenus at 4293 dollars which is a little over 50% of the total, with Merch, Drink, and RainGear trailing far behind.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values('mean', ascending=False)
by_vendor

,mean,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


VV-01 has the highest value average order revenue at 22.60 dollars, but it is close to the V-18's 21.75 dollars and the since both are based on about 100 orders, the gap is samll enough that it is not a decisive difference.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / total_revenue * 100
print(f"Merch share of revenue: {merch_share:.1f}%")

Merch share of revenue: 20.8%


Merch is about over 20% of total revenue which amkes it the second hightest cateogry behidn food.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df)
assert abs(joined['revenue'].sum() - total_revenue) < 0.01

unmatched = df.loc[~df['vendor_id'].isin(vendor_names['vendor_id']), 'vendor_id'].unique()
print("Unmatched vendor:", unmatched)
print("Its order count:", (df['vendor_id'] == unmatched[0]).sum())
print("Its revenue:", df.loc[df['vendor_id'] == unmatched[0], 'revenue'].sum())

Unmatched vendor: ['V-18']
Its order count: 108
Its revenue: 2349.0


The left join kept all 400 rows and the full 8520 revenue total confriming the valideate function "many to one" passed without having to make duplicates or dropping any of the orders.

**The unmatched vendor, and what I did about it:** V-18 has no entry in the vendor name loookup so it came up as NaN in the vendor name after the left join. I kept it in the merged table rather than having to drop it since V-18 is actually the largest reveneu vendor and keepign it out would cuase distortions.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pivot = pd.pivot_table(
    df,
    index='vendor_id',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


The pivot table will show each category combination with row and column totals. This makes it easy to see that Food is every vendor's largest category and V-18 leads in overall revenue.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. Vendors should go ahead and stock more Food and less rainGear next game. Food generated by itself was about 4293 dollars while RainGear brought in only aobut 901.50 dollars which is a comparison of about 50% to 10%... suggesting that RainGear inventory is oversized and relative to what the demand is looking like.

b. Q3 which is on the average order revenue by vendor is the least trustworthy answer because the vendor averages are all clustered tightly between a 20.31 to 22.60 dollars range on sample sizes of roughly 90-110 orders each. This means that the 'highest' vendor could easily flip wuth a different random draw rather than being a good interpretation of a real difference in the vendors performance.